# 🛣️ JalanCerdas AI — YOLO Training (Google Colab GPU)

Training model YOLOv8n untuk deteksi kerusakan jalan (6 kelas).

**⚠️ Penting:** Jalankan notebook ini dengan **GPU runtime**!

```
Runtime → Change runtime type → T4 GPU
```

## 1️⃣ Setup & Install Dependencies

In [ ]:
!pip install -q ultralytics kagglehub pyyaml tqdm

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    print("NO GPU! Go to Runtime > Change runtime type > GPU")

## 2️⃣ Download Dataset

In [ ]:
import kagglehub
import os

print("Downloading dataset...")
path = kagglehub.dataset_download("habibiahmadim09/kerusakan-jalan")
print(f"Downloaded to: {path}")

DATASET_SRC = os.path.join(path, "kerusakan-jalan")
print(f"Dataset source: {DATASET_SRC}")
for split in ["train", "valid", "test"]:
    split_path = os.path.join(DATASET_SRC, split)
    if os.path.exists(split_path):
        files = os.listdir(split_path)
        images = [f for f in files if f.endswith((".jpg", ".jpeg", ".png"))]
        print(f"  {split}: {len(images)} images")

## 3️⃣ Convert VOC XML to YOLO

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

CLASS_NAMES = [
    "retak_memanjang",
    "pengelupasan_lapisan_permukaan",
    "lubang",
    "retak_kulit_buaya",
    "retak_blok",
    "retak_pinggir",
]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

print("Class mapping:")
for name, cid in CLASS_TO_ID.items():
    print(f"  {cid}: {name}")


def voc_to_yolo(xml_path, class_to_id):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    img_w = int(size.find("width").text)
    img_h = int(size.find("height").text)
    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in class_to_id:
            continue
        class_id = class_to_id[name]
        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)
        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h
        x_center = max(0, min(1, x_center))
        y_center = max(0, min(1, y_center))
        width = max(0, min(1, width))
        height = max(0, min(1, height))
        lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    return lines


DATASET_DST = "./dataset"
class_counts = defaultdict(int)
total_images = 0
total_labels = 0

for split in ["train", "valid", "test"]:
    src_split = Path(DATASET_SRC) / split
    if not src_split.exists():
        continue
    dst_split_name = "val" if split == "valid" else split
    dst_images = Path(DATASET_DST) / dst_split_name / "images"
    dst_labels = Path(DATASET_DST) / dst_split_name / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    images = [f for f in src_split.iterdir() if f.suffix.lower() in img_exts]
    print(f"\nProcessing {split} ({len(images)} images)...")
    for img_file in images:
        shutil.copy2(img_file, dst_images / img_file.name)
        total_images += 1
        xml_file = img_file.with_suffix(".xml")
        if xml_file.exists():
            yolo_lines = voc_to_yolo(xml_file, CLASS_TO_ID)
            if yolo_lines:
                txt_file = dst_labels / (img_file.stem + ".txt")
                txt_file.write_text("\n".join(yolo_lines))
                total_labels += len(yolo_lines)
                for line in yolo_lines:
                    class_id = int(line.split()[0])
                    class_counts[CLASS_NAMES[class_id]] += 1
    print(f"  Done")

print(f"\nTotal: {total_images} images, {total_labels} labels")
for name, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {name}: {count}")

## 4️⃣ Create data.yaml

In [ ]:
%%writefile data.yaml
# JalanCerdas AI - YOLO Config
path: ./dataset
train: train/images
val: val/images
test: test/images
nc: 6
names:
  0: retak_memanjang
  1: pengelupasan_lapisan_permukaan
  2: lubang
  3: retak_kulit_buaya
  4: retak_blok
  5: retak_pinggir

In [ ]:
import os
WORK_DIR = "/content"  # Colab working directory
DATA_YAML = os.path.join(WORK_DIR, "data.yaml")
WEIGHTS_DIR = os.path.join(WORK_DIR, "runs", "train", "weights")
BEST_PT = os.path.join(WEIGHTS_DIR, "best.pt")

for split in ["train", "val", "test"]:
    img_dir = os.path.join(WORK_DIR, "dataset", split, "images")
    lbl_dir = os.path.join(WORK_DIR, "dataset", split, "labels")
    if os.path.exists(img_dir):
        imgs = len([f for f in os.listdir(img_dir) if f.endswith((".jpg", ".png"))])
        lbls = len([f for f in os.listdir(lbl_dir) if f.endswith(".txt")]) if os.path.exists(lbl_dir) else 0
        print(f"  {split}: {imgs} images, {lbls} labels")
    else:
        print(f"  {split}: NOT FOUND")

print(f"\nData YAML: {DATA_YAML}")
print(f"Weights will be saved to: {WEIGHTS_DIR}")

## 5️⃣ Train YOLOv8n (100 Epochs, GPU)

In [ ]:
from ultralytics import YOLO
import torch
import os

WORK_DIR = "/content"
DATA_YAML = os.path.join(WORK_DIR, "data.yaml")

device = "0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\nLoading YOLOv8n...")
model = YOLO("yolov8n.pt")

print("\nStarting training (100 epochs)...")
print("This will take ~1.5-4 hours on GPU\n")

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    project=os.path.join(WORK_DIR, "runs"),
    name="train",
    exist_ok=True,
    patience=20,
    save=True,
    save_period=10,
    workers=4,
    verbose=True,
)

BEST_PT = os.path.join(WORK_DIR, "runs", "train", "weights", "best.pt")
LAST_PT = os.path.join(WORK_DIR, "runs", "train", "weights", "last.pt")
print(f"\nTraining done!")
print(f"Best model: {BEST_PT} (exists: {os.path.exists(BEST_PT)})")
print(f"Last model: {LAST_PT} (exists: {os.path.exists(LAST_PT)})")

## 6️⃣ Evaluate Model

In [ ]:
from ultralytics import YOLO
import torch
import os

WORK_DIR = "/content"
DATA_YAML = os.path.join(WORK_DIR, "data.yaml")
BEST_PT = os.path.join(WORK_DIR, "runs", "train", "weights", "best.pt")

# Find the correct weights path
if not os.path.exists(BEST_PT):
    # Try alternative paths
    alt_paths = [
        os.path.join(WORK_DIR, "runs", "train", "weights", "best.pt"),
        os.path.join(WORK_DIR, "runs", "detect", "train", "weights", "best.pt"),
    ]
    for p in alt_paths:
        if os.path.exists(p):
            BEST_PT = p
            break
    else:
        print("WARNING: best.pt not found. Searching...")
        for root, dirs, files in os.walk(os.path.join(WORK_DIR, "runs")):
            if "best.pt" in files:
                BEST_PT = os.path.join(root, "best.pt")
                print(f"Found: {BEST_PT}")
                break

print(f"Loading model: {BEST_PT}")
print(f"File exists: {os.path.exists(BEST_PT)}")

model = YOLO(BEST_PT)

print("\nEvaluating on validation set...")
metrics = model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=16,
    device="0" if torch.cuda.is_available() else "cpu",
)

print(f"\nResults:")
print(f"  mAP50:    {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall:    {metrics.box.mr:.4f}")

class_names = ["retak_memanjang", "pengelupasan_lapisan_permukaan", "lubang", "retak_kulit_buaya", "retak_blok", "retak_pinggir"]
print(f"\nPer-class mAP50:")
for i, (name, ap) in enumerate(zip(class_names, metrics.box.maps)):
    print(f"  {i}. {name}: {ap:.4f}")

## 7️⃣ Export to TFLite

In [ ]:
from ultralytics import YOLO
import os

WORK_DIR = "/content"
BEST_PT = os.path.join(WORK_DIR, "runs", "train", "weights", "best.pt")

if not os.path.exists(BEST_PT):
    for root, dirs, files in os.walk(os.path.join(WORK_DIR, "runs")):
        if "best.pt" in files:
            BEST_PT = os.path.join(root, "best.pt")
            break

print(f"Loading: {BEST_PT}")
model = YOLO(BEST_PT)

print("\nExporting to TFLite...")
model.export(format="tflite", imgsz=640, half=True, int8=False, nms=True)

tflite_path = BEST_PT.replace(".pt", ".tflite")
if os.path.exists(tflite_path):
    size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
    print(f"\nTFLite exported: {tflite_path}")
    print(f"Size: {size_mb:.2f} MB")
else:
    print("Export failed - check errors above")

## 8️⃣ Download Model

In [ ]:
import os
import shutil
from google.colab import files

WORK_DIR = "/content"
WEIGHTS_DIR = os.path.join(WORK_DIR, "runs", "train", "weights")
output_dir = "jalanCerdas_model"
os.makedirs(output_dir, exist_ok=True)

model_files = {
    "best.pt": "best.pt",
    "best.tflite": "best.tflite",
    "best.onnx": "best.onnx",
}

# Also check for training results
results_csv = os.path.join(WORK_DIR, "runs", "train", "results.csv")
if os.path.exists(results_csv):
    model_files["results.csv"] = "training_results.csv"

print("Model files:")
for src_name, dst_name in model_files.items():
    src_path = os.path.join(WEIGHTS_DIR, src_name) if src_name.startswith("best") else src_path
    if src_name == "results.csv":
        src_path = results_csv
    else:
        src_path = os.path.join(WEIGHTS_DIR, src_name)
    if os.path.exists(src_path):
        shutil.copy2(src_path, os.path.join(output_dir, dst_name))
        size = os.path.getsize(src_path) / (1024 * 1024)
        print(f"  {dst_name} ({size:.2f} MB)")
    else:
        print(f"  {dst_name} (not found)")

shutil.make_archive(output_dir, 'zip', '.', output_dir)
zip_size = os.path.getsize(f"{output_dir}.zip") / (1024 * 1024)
print(f"\nZipped: {output_dir}.zip ({zip_size:.2f} MB)")
print("\nDownloading...")
files.download(f"{output_dir}.zip")

## 9️⃣ Test Prediction

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

WORK_DIR = "/content"
BEST_PT = os.path.join(WORK_DIR, "runs", "train", "weights", "best.pt")

if not os.path.exists(BEST_PT):
    for root, dirs, files in os.walk(os.path.join(WORK_DIR, "runs")):
        if "best.pt" in files:
            BEST_PT = os.path.join(root, "best.pt")
            break

print(f"Loading: {BEST_PT}")
model = YOLO(BEST_PT)

val_dir = os.path.join(WORK_DIR, "dataset", "val", "images")
if os.path.exists(val_dir):
    val_images = [f for f in os.listdir(val_dir) if f.endswith((".jpg", ".png"))]
    print(f"Found {len(val_images)} validation images")
    for i, img_name in enumerate(val_images[:5]):
        img_path = os.path.join(val_dir, img_name)
        results = model.predict(source=img_path, imgsz=640, conf=0.5, save=True, project=os.path.join(WORK_DIR, "runs"), name="predict", exist_ok=True)
        print(f"\nImage {i+1}: {img_name}")
        for r in results:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                class_name = model.names[cls_id]
                print(f"  {class_name}: {conf:.2%}")
    pred_img = os.path.join(WORK_DIR, "runs", "predict", val_images[0])
    if os.path.exists(pred_img):
        print(f"\nFirst prediction:")
        display(Image(pred_img, width=600))
else:
    print("No validation images found")